In [1]:
import googlemaps
import sqlite3
from datetime import datetime
import sys
import os
import importlib.util

# Absoluten Pfad zum gewünschten Ordner ermitteln
notebook_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "hidden"))

# Pfad zu sys.path hinzufügen
if notebook_dir not in sys.path:
    sys.path.append(notebook_dir)

# Modul dynamisch importieren
auth_path = os.path.join(notebook_dir, "auth.py")
spec = importlib.util.spec_from_file_location("auth", auth_path)
auth = importlib.util.module_from_spec(spec)
spec.loader.exec_module(auth)

from auth import google_auth # Personal Google API key is saved separately

In [2]:
conn = sqlite3.connect(r"..\data\processed\plz.sqlite")
cur = conn.cursor()

In [3]:
# Loads personalGgoogle API key and initializes googlemaps Client with it
auth_google = google_auth()
gmaps = googlemaps.Client(key=auth_google)

In [4]:
# cur.execute("""SELECT id, plz, lat, lng FROM Plz WHERE google_retrieved = 0""")
cur.execute(""" SELECT id, plz, lat, lng 
                FROM Plz 
                WHERE google_retrieved = 0 
                AND (plz LIKE '71%' OR plz LIKE '07%' OR plz LIKE '08%')""") #(plz LIKE '8%' OR plz LIKE '9%' OR plz LIKE '63%')
plzs = cur.fetchmany(1436)
# plzs = cur.fetchall()

In [6]:
len(plzs)

82

In [7]:
# Sets the destinations all (clinica) and the depature time
destinations = ["LMU Klinikum Campus Großhadern Marchioninistraße 15, 81377 München, Germany",
                "Klinikum rechts der Isar der Technischen Universität München Ismaninger Str. 22, 81675 München, Germany",
                "Universitätsklinikum Würzburg Josef-Schneider-Straße 2, 97080 Würzburg, Germany",
                "Universitätsklinikum Erlangen Maximilianspl. 2, 91054 Erlangen, Germany",
                "Universitätsklinikum Regensburg Franz-Josef-Strauß-Allee 11, 93053 Regensburg, Germany",
                "Universitätsklinikum Augsburg Stenglinstraße 2, 86156 Augsburg, Germany"
                ]

departure_time = datetime(2025,3,13,7,0,0,0)
depature_time_str = departure_time.strftime("%Y/%m/%d, %H:%M")


In [8]:
print(depature_time_str)

2025/03/13, 07:00


In [9]:
# If no valid travel is returned by Google (ZERO_RESULTS) the value "None" is returned, else the travel distance value
def distance(results, plz, nr):
    if results["rows"][0]["elements"][nr]["status"] == "ZERO_RESULTS":
        print(plz, " without results for #", nr)
        return None
    elif results["rows"][0]["elements"][nr]["status"] == "OK":
        return results["rows"][0]["elements"][nr]["distance"]["value"]
    else:
        print("Unknown status:")
        print(results["rows"][0]["elements"][nr]["status"])

In [10]:
# If no valid travel is returned by Google (ZERO_RESULTS) the value "None" is returned, else the travel time value
def time(results, plz, nr):
    if results["rows"][0]["elements"][nr]["status"] == "ZERO_RESULTS":
        print(plz, " without results for #", nr)
        return None
    elif results["rows"][0]["elements"][nr]["status"] == "OK":
        return results["rows"][0]["elements"][nr]["duration"]["value"]
    else:
        print("Unknown status:")
        print(results["rows"][0]["elements"][nr]["status"])

In [11]:
# Loops over all retrieved communities and sends request to google maps distance_matrix (1 plz as origin and all clinica as destination)
# Gets results for travel by car and travel by transit.
# Parses the returned results (JSON Format) using functions "distance" and "time" (see above) and updates the database &sets google_retrieved = 1
k = 0
for plz in plzs:

    k += 1
    #origins = plz[1] + " " + plz[2] + ", Germany"
    origins = str(plz[2]) + "," + str(plz[3])
    print(k, ":",plz[1], origins)    

    results_car = gmaps.distance_matrix(origins=origins, destinations=destinations, departure_time=departure_time, mode="driving", traffic_model="best_guess")
    results_transit = gmaps.distance_matrix(origins=origins, destinations=destinations, departure_time=departure_time, mode="transit")
    
    car_distance_lmu = distance(results_car, plz, 0)
    car_time_lmu = time(results_car, plz, 0)
    car_distance_tum = distance(results_car, plz, 1)
    car_time_tum = time(results_car, plz, 1)
    car_distance_würzburg = distance(results_car, plz, 2)
    car_time_würzburg = time(results_car, plz, 2)
    car_distance_erlangen = distance(results_car, plz, 3)
    car_time_erlangen = time(results_car, plz, 3)
    car_distance_regensburg = distance(results_car, plz, 4)
    car_time_regensburg = time(results_car, plz, 4)
    car_distance_augsburg = distance(results_car, plz, 5)
    car_time_augsburg = time(results_car, plz, 5)

    transit_distance_lmu = distance(results_transit, plz, 0)
    transit_time_lmu = time(results_transit, plz, 0)
    transit_distance_tum = distance(results_transit, plz, 1)
    transit_time_tum = time(results_transit, plz, 1)
    transit_distance_würzburg = distance(results_transit, plz, 2)
    transit_time_würzburg = time(results_transit, plz, 2)
    transit_distance_erlangen = distance(results_transit, plz, 3)
    transit_time_erlangen = time(results_transit, plz, 3)
    transit_distance_regensburg = distance(results_transit, plz, 4)
    transit_time_regensburg = time(results_transit, plz, 4)
    transit_distance_augsburg = distance(results_transit, plz, 5)
    transit_time_augsburg = time(results_transit, plz, 5)

    cur.execute("""UPDATE Plz SET 
                        car_distance_lmu =?,             /*1*/
                        car_time_lmu =?,
                        car_distance_tum =?,
                        car_time_tum =?,
                        car_distance_würzburg =?,        /*5*/
                        car_time_würzburg =?,
                        car_distance_erlangen =?,
                        car_time_erlangen =?,
                        car_distance_regensburg =?,
                        car_time_regensburg =?,          /*10*/
                        car_distance_augsburg =?,
                        car_time_augsburg =?,
                        transit_distance_lmu =?,
                        transit_time_lmu =?,
                        transit_distance_tum =?,         /*15*/
                        transit_time_tum =?,
                        transit_distance_würzburg =?,
                        transit_time_würzburg =?,
                        transit_distance_erlangen =?,
                        transit_time_erlangen =?,        /*20*/
                        transit_distance_regensburg =?,
                        transit_time_regensburg =?,
                        transit_distance_augsburg =?,
                        transit_time_augsburg =?,
                        google_retrieved =1,             /*25*/
                        date_calc = ?
                    WHERE id=?""",
                    (   car_distance_lmu,                #1
                        car_time_lmu,
                        car_distance_tum,
                        car_time_tum,
                        car_distance_würzburg,           #5
                        car_time_würzburg,
                        car_distance_erlangen,
                        car_time_erlangen,
                        car_distance_regensburg,
                        car_time_regensburg,             #10
                        car_distance_augsburg,
                        car_time_augsburg,
                        transit_distance_lmu,
                        transit_time_lmu,
                        transit_distance_tum,            #15
                        transit_time_tum,
                        transit_distance_würzburg,
                        transit_time_würzburg,
                        transit_distance_erlangen,
                        transit_time_erlangen,           #20
                        transit_distance_regensburg,
                        transit_time_regensburg,
                        transit_distance_augsburg,
                        transit_time_augsburg,
                                                                        #25
                        depature_time_str,
                    plz[0]))         
    # print(origins)
conn.commit()                         



1 : 71032 48.6802429,9.055195
2 : 71034 48.6762593,8.9773145
3 : 71063 48.7131518,9.0000948
4 : 71065 48.71082260000001,9.0522388
5 : 71067 48.7252788,9.010019
6 : 71069 48.70781059999999,8.9618345
7 : 71083 48.5879291,8.879328
8 : 71088 48.6427688,9.0176454
9 : 71093 48.613717,9.0657204
10 : 71101 48.6595376,9.061313
11 : 71106 48.7473056,8.9744064
12 : 71111 48.6338945,9.129683199999999
13 : 71116 48.641659,8.9026297
14 : 71120 48.7144885,8.9121197
15 : 71126 48.5558483,8.837527500000002
16 : 71131 48.5694299,8.7785929
17 : 71134 48.6884647,8.8855667
18 : 71139 48.6584785,8.9461595
19 : 71144 48.6654418,9.1143327
20 : 71149 48.5263115,8.8282771
21 : 71154 48.6219403,8.8964043
22 : 71155 48.6033543,9.0050434
(5044, '71155', 48.6033543, 9.0050434)  without results for # 0
(5044, '71155', 48.6033543, 9.0050434)  without results for # 0
(5044, '71155', 48.6033543, 9.0050434)  without results for # 1
(5044, '71155', 48.6033543, 9.0050434)  without results for # 1
(5044, '71155', 48.603354

In [60]:
# # cur.execute("""SELECT id, plz, lat, lng FROM Plz WHERE google_retrieved = 0""")
# cur.execute(""" SELECT id, plz, lat, lng 
#                 FROM Plz 
#                 WHERE google_retrieved = 1 
#                 AND transit_distance_erlangen IS NULL
#                 AND plz = 36275""") #(plz LIKE '8%' OR plz LIKE '9%' OR plz LIKE '63%')
# plz_sel = cur.fetchmany(10)
# # plzs = cur.fetchall()

In [35]:
cur.close()
conn.close()